# 2. DB 전용 운영 노트북

동일한 독립 크롤러를 사용하되 **메뉴 CSV를 출력하지 않고 데이터베이스만 갱신**하는 재현 예시입니다.

- `MODE='demo'`: 합성 HTML + 격리된 로컬 SQLite. 프로덕션 DB 접근 없음.
- `MODE='live'`: 실제 외부 사이트에 요청, 환경변수 `DATABASE_URL`에 지정한 DB에 적재합니다.
- 실제 앱 운영은 이 노트북 대신 `python -m chajwo_meals.db_runner --live --days 30`과 **영속 PostgreSQL**을 사용하세요.
- 성공한 출처·날짜만 교체(upsert)하고, 실패/미게시 날짜는 유지합니다. 오류 및 실행이력은 DB의 `crawl_errors`, `crawl_runs` 테이블에 누적합니다.

In [ ]:
from pathlib import Path
import sys, json, sqlite3, os
from datetime import date, datetime
from zoneinfo import ZoneInfo
HERE=Path.cwd().resolve()
ROOT=next((d for d in (HERE, HERE.parent, HERE.parent.parent) if (d/'src/chajwo_meals').is_dir()),None)
if ROOT is None:
    raise RuntimeError('프로젝트 루트 또는 notebooks 폴더에서 실행하세요.')
sys.path.insert(0,str(ROOT/'src'))
print('ROOT:', ROOT)


## 2. DB 연결 설정

In [ ]:
MODE='demo' # 실제 운영 검증 시에만 'live'로 변경
DAYS=1 if MODE=='demo' else 30
SOURCES=('snuco','snudorm','vet')
START_DATE=date(2026,10,22) if MODE=='demo' else datetime.now(ZoneInfo('Asia/Seoul')).date()
if MODE=='demo':
    DB_PATH=ROOT/'outputs'/'db_demo_only'/'meals.sqlite3'
    DB_URL='sqlite:///'+str(DB_PATH)
else:
    DB_PATH=None
    DB_URL=os.environ.get('DATABASE_URL') or os.environ.get('CHAJWO_DATABASE_URL')
    if not DB_URL:
        raise RuntimeError('live 모드 DB_URL 없음: DATABASE_URL 환경변수를 설정하세요.')
print('모드:',MODE,'일수:',DAYS,'DB 타입:',('PostgreSQL' if DB_URL.startswith(('postgresql://','postgres://')) else 'SQLite'))


## 3. 테스트 HTML (demo 모드 전용)

In [ ]:
from urllib.parse import urlsplit, parse_qs
class FixtureFetcher:
    """실제 네트워크 대신 자체 작성한 테스트 HTML/JSON을 반환합니다."""
    def __init__(self, root):
        self.root=root
        self.requests=[]
    def get(self, url):
        self.requests.append(url)
        if 'snuco.snu.ac.kr/foodmenu/' in url:
            name='snuco.html'
        elif 'snudorm.snu.ac.kr/foodmenu/' in url:
            name='snudorm.html'
        elif 'vet.snu.ac.kr/cafe_menu/' in url:
            name='vet.html'
        elif '/식당안내/' in url:
            name='guide_demo.html'
        elif 'map.snu.ac.kr/api/building.action' in url:
            name='campus_map_demo.json'
        else:
            raise ValueError(f'예상하지 않은 요청: {url}')
        html=(self.root/name).read_text(encoding='utf-8')
        if name in ('snuco.html','snudorm.html'):
            day=parse_qs(urlsplit(url).query)['date'][0]
            html=html.replace('2026-10-22', day)
        return html


## 4. 식단 및 식당 정보 수집

In [ ]:
from chajwo_meals.pipeline import collect, collect_metadata
from chajwo_meals.http import RateLimitedFetcher
fetcher=FixtureFetcher(ROOT/'tests'/'fixtures') if MODE=='demo' else RateLimitedFetcher()
rows, errors = collect(start=START_DATE, days=DAYS, sources=SOURCES, fetcher=fetcher)
restaurant_directory, metadata_errors=collect_metadata(fetcher=fetcher)
errors.extend(metadata_errors)
print('원문 메뉴 줄 수:',len(rows),'식당/코너 수:',len(restaurant_directory),'오류 수:',len(errors))
if errors: print('오류 예시:',errors[:3])


## 5. DB에만 동기화

In [ ]:
from chajwo_meals.pipeline import flatten_entries, summarize
from chajwo_meals.fixed import fixed_item_rows, load_fixed_catalog
from chajwo_meals.hours import extract_service_hours
from chajwo_meals.database import sync_db

# 고정 메뉴는 별도 카탈로그에서 1회만 반영합니다.
entries=flatten_entries([r for r in rows if not (r['source']=='snuco' and r['restaurant'] in ('버거운버거','공대간이식당'))])
if 'snuco' in SOURCES:
    entries.extend({'entry_kind':'fixed_catalog', **r} for r in fixed_item_rows(load_fixed_catalog()))
manifest=summarize(rows,errors,started=START_DATE,days=DAYS,sources=SOURCES,restaurant_directory=restaurant_directory)
manifest['storage']='db'
result=sync_db(DB_URL,entries=entries,directory=restaurant_directory,errors=errors,
               started=START_DATE.isoformat(),days=DAYS,manifest=manifest,
               hour_rows=extract_service_hours(rows))
print('DB 동기화 결과:',json.dumps(result,ensure_ascii=False,indent=2))
print('CSV 내보내기 함수는 호출하지 않았습니다.')


## 6. 저장 결과 검사 (데모 SQLite만 직접 읽기)

In [ ]:
if MODE=='demo':
    with sqlite3.connect(DB_PATH) as conn:
        for table in ('restaurants','menu_entries','meal_service_hours','crawl_runs','crawl_errors'):
            print(table,conn.execute('SELECT COUNT(*) FROM '+table).fetchone()[0])
        print('호구세트 가격:', conn.execute("SELECT price_won FROM menu_entries WHERE menu_name='호구세트' AND data_origin='fixed_catalog'").fetchone()[0])
    assert DB_PATH.exists()
else:
    print('라이브 DB는 앱 백엔드에서 read-only 쿼리로 결과 확인 권장. 비밀번호를 노트북에 출력하지 마세요.')


## 7. 앱 서버에서 읽는 쿼리 예시

```sql
SELECT restaurant_unit, menu_name, price_won, source_date, meal_type, data_origin
FROM menu_entries
WHERE source_date = '2026-10-22' OR data_origin = 'fixed_catalog'
ORDER BY restaurant_unit, meal_type, menu_name;
```

`restaurants` 테이블은 건물/좌표/일반 운영시간, `meal_service_hours`는 특정 날짜별 운영 공지입니다. 앱에는 DB에 직접 연결하는 비밀번호를 넣지 말고 **백엔드 API에서 조회**하도록 연결하세요.